In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from gaze_cam.dataset import make_loaders
from gaze_cam.gaze_utils import load_gaze_file, slice_gaze_for_clip
from scripts.analysis import clip_alignment_metrics

In [3]:
def center_gaussian(h, w, sigma=0.25):
    yy, xx = np.mgrid[0:h, 0:w]
    cy = (h - 1) / 2
    cx = (w - 1) / 2

    sigma = sigma * min(h, w)

    g = np.exp(-((xx - cx)**2 + (yy - cy)**2)/(2*sigma**2))
    g = g / g.max()
    return g

In [26]:
MODEL_CONFIG = {
    "r3d18": (16, 112, 112),
    "slowfast_r50": (32, 224, 224),
    "timesformer": (8, 224, 224),
    "vivit": (32, 224, 224),
}

In [8]:
_, test_loader, _, _ = make_loaders(split=1, arch="r3d18")

[dataset] kept=8299  miss_no_file=0  miss_parse=0  miss_label=0
[dataset] kept=2022  miss_no_file=0  miss_parse=0  miss_label=0
[cache] r3d18: caching 8299/8299 clips ...


caching r3d18: 100%|██████████| 8299/8299 [33:26<00:00,  4.14it/s]  


[cache] r3d18: done
[cache] r3d18: caching 2022/2022 clips ...


caching r3d18: 100%|██████████| 2022/2022 [07:56<00:00,  4.24it/s]

[cache] r3d18: done
[r3d18] Train items: 8299  Test items: 2022  Num actions: 1170  Frames: 16  Size: 112


ValueError: too many values to unpack (expected 4)

In [ ]:
def make_center_gaussian(h, w, sigma_frac=0.25):
    yy, xx = np.mgrid[0:h, 0:w]
    cy = (h - 1) / 2.0
    cx = (w - 1) / 2.0
    sigma = sigma_frac * min(h, w)

    g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2))
    g = g / (g.max() + 1e-8)
    return g.astype(np.float32)

def make_center_cam(T, H, W, sigma_frac=0.25):
    g = make_center_gaussian(H, W, sigma_frac=sigma_frac)
    cam = np.repeat(g[None, :, :], T, axis=0)   # (T, H, W)
    return cam

In [18]:
def run_center_bias_baseline(model_name, split=1, max_test=0, sigma_frac=0.25, verbose=False):
    T, H, W = MODEL_CONFIG[model_name]

    _, test_loader, _, _, _ = make_loaders(
        split=split,
        arch=model_name,
        max_test=max_test if max_test > 0 else None
    )

    center_cam = make_center_cam(T, H, W, sigma_frac=sigma_frac)
    rows = []

    for xs, ys, paths, metas in tqdm(test_loader, desc=f"center-bias {model_name}"):
        for i, meta in enumerate(metas):
            try:
                session = meta["session"]
                f0 = int(meta["f0"])
                f1 = int(meta["f1"])
                stem = meta.get("stem", str(paths[i]))

                gaze_df = load_gaze_file(session)
                gaze_clip = slice_gaze_for_clip(gaze_df, f0, f1)

                # skip empty or unusable gaze clips
                if gaze_clip is None or len(gaze_clip) == 0:
                    if verbose:
                        print(f"[SKIP empty gaze] {model_name} | {stem} | session={session} | f0={f0} | f1={f1}")
                    continue

                if "x" not in gaze_clip.columns or "y" not in gaze_clip.columns:
                    if verbose:
                        print(f"[SKIP no x/y] {model_name} | {stem}")
                    continue

                valid_xy = gaze_clip["x"].notna() & gaze_clip["y"].notna()
                if valid_xy.sum() == 0:
                    if verbose:
                        print(f"[SKIP no valid xy] {model_name} | {stem}")
                    continue

                metrics = clip_alignment_metrics(
                    cam_thw=center_cam,
                    gaze_df=gaze_clip,
                    f0=f0,
                    f1=f1,
                    frame_h=H,
                    frame_w=W,
                    temporal_shift=0,
                    compute_auc=True,
                )

                rows.append({
                    "model": model_name,
                    "stem": stem,
                    "path": paths[i],
                    "session": session,
                    "f0": f0,
                    "f1": f1,
                    "nss": metrics["nss_mean"],
                    "auc": metrics["auc_mean"],
                    "valid_frames": metrics["valid_frames"],
                    "total_frames": metrics["total_frames"],
                })

            except Exception as e:
                if verbose:
                    print(f"[SKIP error] {model_name} | path={paths[i]} | err={e}")
                continue

    return pd.DataFrame(rows)

In [20]:
def run_center_bias_baseline(model_name, split=1, max_test=0, sigma_frac=0.25, verbose=True):
    T, H, W = MODEL_CONFIG[model_name]

    _, test_loader, _, _, _ = make_loaders(
        split=split,
        arch=model_name,
        max_test=max_test if max_test > 0 else None
    )

    center_cam = make_center_cam(T, H, W, sigma_frac=sigma_frac)
    rows = []

    skipped_empty_gaze = 0
    skipped_bad_meta = 0
    skipped_metric_fail = 0

    for xs, ys, paths, metas in tqdm(test_loader, desc=f"center-bias {model_name}"):
        for i, meta in enumerate(metas):
            try:
                session = meta["session"]
                f0 = int(meta["f0"])
                f1 = int(meta["f1"])
                stem = meta.get("stem", str(paths[i]))
            except Exception as e:
                skipped_bad_meta += 1
                if verbose:
                    print(f"[SKIP bad meta] path={paths[i]} meta={meta} err={e}")
                continue

            try:
                gaze_df = load_gaze_file(session)
                gaze_clip = slice_gaze_for_clip(gaze_df, f0, f1)

                if gaze_clip is None or len(gaze_clip) == 0:
                    skipped_empty_gaze += 1
                    if verbose:
                        print(f"[SKIP empty gaze] session={session}, stem={stem}, f0={f0}, f1={f1}")
                    continue

                metrics = clip_alignment_metrics(
                    cam_thw=center_cam,
                    gaze_df=gaze_clip,
                    f0=f0,
                    f1=f1,
                    frame_h=H,
                    frame_w=W,
                    temporal_shift=0,
                    compute_auc=True,
                )

                rows.append({
                    "model": model_name,
                    "stem": stem,
                    "path": paths[i],
                    "session": session,
                    "f0": f0,
                    "f1": f1,
                    "nss": metrics["nss_mean"],
                    "auc": metrics["auc_mean"],
                    "valid_frames": metrics["valid_frames"],
                    "total_frames": metrics["total_frames"],
                })

            except Exception as e:
                skipped_metric_fail += 1
                if verbose:
                    print(f"[SKIP metric fail] stem={stem}, session={session}, f0={f0}, f1={f1}, err={e}")
                continue

    df = pd.DataFrame(rows)

    print(f"\n[{model_name}] done")
    print(f"kept clips          : {len(df)}")
    print(f"skipped empty gaze  : {skipped_empty_gaze}")
    print(f"skipped bad meta    : {skipped_bad_meta}")
    print(f"skipped metric fail : {skipped_metric_fail}")

    return df

In [27]:
df_r3d = run_center_bias_baseline("r3d18", split=1, max_test=50)
df_r3d[["nss", "auc"]].mean()

[dataset] kept=8299  miss_no_file=0  miss_parse=0  miss_label=0
[dataset] kept=50  miss_no_file=0  miss_parse=0  miss_label=0
[cache] r3d18: all 8299 clips already cached
[cache] r3d18: all 50 clips already cached
[r3d18] Train items: 8299  Test items: 50  Num actions: 1170  Frames: 16  Size: 112


center-bias r3d18:   0%|          | 0/4 [00:00<?, ?it/s]c:\Users\maya\Desktop\mosaic\.conda\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
center-bias r3d18: 100%|██████████| 4/4 [00:11<00:00,  2.83s/it]


[r3d18] done
kept clips          : 50
skipped empty gaze  : 0
skipped bad meta    : 0
skipped metric fail : 0


nss    1.034345
auc    0.763538
dtype: float64

In [31]:
df_all = pd.concat(all_dfs, ignore_index=True)

summary = (
    df_all.groupby("model")[["nss", "auc", "valid_frames"]]
    .mean()
    .sort_index()
)

summary

,nss,auc,valid_frames
model,,,
r3d18,0.873822,0.731667,92.078921
slowfast_r50,0.885506,0.734026,92.078921


In [33]:
def run_center_bias_baseline(model_name, split=1, max_test=0, sigma_frac=0.25, verbose=True):
    T, H, W = MODEL_CONFIG[model_name]
    T, H, W = int(T), int(H), int(W)

    # use a loader that works, only to get clips + meta
    _, test_loader, _, _, _ = make_loaders(
        split=split,
        arch="r3d18",
        max_test=max_test if max_test > 0 else None
    )

    center_cam = make_center_cam(T, H, W, sigma_frac=sigma_frac)
    rows = []

    skipped = 0

    for xs, ys, paths, metas in tqdm(test_loader, desc=f"center-bias {model_name}"):
        for i, meta in enumerate(metas):
            try:
                session = meta["session"]
                f0 = int(meta["f0"])
                f1 = int(meta["f1"])
                stem = meta.get("stem", str(paths[i]))

                gaze_df = load_gaze_file(session)
                gaze_clip = slice_gaze_for_clip(gaze_df, f0, f1)

                if gaze_clip is None or len(gaze_clip) == 0:
                    skipped += 1
                    continue

                if "x" not in gaze_clip.columns or "y" not in gaze_clip.columns:
                    skipped += 1
                    continue

                valid_xy = gaze_clip["x"].notna() & gaze_clip["y"].notna()
                if valid_xy.sum() == 0:
                    skipped += 1
                    continue

                metrics = clip_alignment_metrics(
                    cam_thw=center_cam,
                    gaze_df=gaze_clip,
                    f0=f0,
                    f1=f1,
                    frame_h=H,
                    frame_w=W,
                    temporal_shift=0,
                    compute_auc=True,
                )

                rows.append({
                    "model": model_name,
                    "stem": stem,
                    "path": paths[i],
                    "session": session,
                    "f0": f0,
                    "f1": f1,
                    "nss": metrics["nss_mean"],
                    "auc": metrics["auc_mean"],
                    "valid_frames": metrics["valid_frames"],
                    "total_frames": metrics["total_frames"],
                })

            except Exception as e:
                skipped += 1
                if verbose:
                    print(f"[SKIP] {model_name} | {paths[i]} | {e}")
                continue

    df = pd.DataFrame(rows)
    print(f"{model_name}: kept={len(df)}, skipped={skipped}")
    return df


In [34]:
# Cell 6: run all 4 models
all_dfs = []

for model_name in ["timesformer", "vivit"]:
    df_model = run_center_bias_baseline(model_name, split=1, max_test=0)
    all_dfs.append(df_model)

df_all = pd.concat(all_dfs, ignore_index=True)

summary = (
    df_all.groupby("model")[["nss", "auc", "valid_frames"]]
    .mean()
    .sort_index()
)

summary

[dataset] kept=8299  miss_no_file=0  miss_parse=0  miss_label=0
[dataset] kept=2022  miss_no_file=0  miss_parse=0  miss_label=0
[cache] r3d18: all 8299 clips already cached
[cache] r3d18: all 2022 clips already cached
[r3d18] Train items: 8299  Test items: 2022  Num actions: 1170  Frames: 16  Size: 112


center-bias timesformer:   0%|          | 0/127 [00:00<?, ?it/s]c:\Users\maya\Desktop\mosaic\.conda\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
center-bias timesformer: 100%|██████████| 127/127 [06:15<00:00,  2.96s/it]


timesformer: kept=2002, skipped=20
[dataset] kept=8299  miss_no_file=0  miss_parse=0  miss_label=0
[dataset] kept=2022  miss_no_file=0  miss_parse=0  miss_label=0
[cache] r3d18: all 8299 clips already cached
[cache] r3d18: all 2022 clips already cached
[r3d18] Train items: 8299  Test items: 2022  Num actions: 1170  Frames: 16  Size: 112


center-bias vivit:   0%|          | 0/127 [00:00<?, ?it/s]c:\Users\maya\Desktop\mosaic\.conda\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
center-bias vivit: 100%|██████████| 127/127 [05:42<00:00,  2.69s/it]

vivit: kept=2002, skipped=20


,nss,auc,valid_frames
model,,,
timesformer,0.885506,0.734026,92.078921
vivit,0.885506,0.734026,92.078921


In [35]:
def make_center_gaussian(h, w, sigma_frac=0.25):
    h = int(h)
    w = int(w)
    yy, xx = np.mgrid[0:h, 0:w]
    cy = (h - 1) / 2.0
    cx = (w - 1) / 2.0
    sigma = float(sigma_frac) * min(h, w)

    g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2))
    g = g / (g.max() + 1e-8)
    return g.astype(np.float32)


def make_center_cam(T, H, W, sigma_frac=0.25):
    T, H, W = int(T), int(H), int(W)
    g = make_center_gaussian(H, W, sigma_frac=sigma_frac)
    cam = np.repeat(g[None, :, :], T, axis=0)   # (T, H, W)
    return cam


In [ ]:
# run one overall center-bias baseline
# 224x224 is a reasonable single baseline choice
df_cb, summary_cb = run_center_bias_baseline(
    split=1,
    max_test=0,          # set to 50 first if you want a quick test
    loader_arch="r3d18", # just use the loader that works
    T=16,                # temporal length barely matters here
    H=224,
    W=224,
    sigma_frac=0.25,
    verbose=False,
)

summary_cb
